In [1]:

# ================================================================
# HindiFlickr30K IAA: Robust Excel header detection + Fleiss' kappa
# Supports title rows, hidden spaces in headings, and multiple sheets
# ================================================================

from pathlib import Path
import re
import unicodedata
import warnings

import numpy as np
import pandas as pd
from statsmodels.stats.inter_rater import fleiss_kappa

# ------------------------------------------------
# 1. EXACT INPUT FILES
# ------------------------------------------------

PATHS = {
    "Annotator1": Path(
        r"D:\DATASET On 10-12-2025\Objective 1\OB1.1\Dataset\KAPPA1\HindiFlickr30K_IAA_Annotator1.xlsx"
    ),
    "Annotator2": Path(
        r"D:\DATASET On 10-12-2025\Objective 1\OB1.1\Dataset\KAPPA1\HindiFlickr30K_IAA_Annotator2.xlsx"
    ),
    "Annotator3": Path(
        r"D:\DATASET On 10-12-2025\Objective 1\OB1.1\Dataset\KAPPA1\HindiFlickr30K_IAA_Annotator3.xlsx"
    ),
}

OUTPUT_DIR = Path(
    r"D:\DATASET On 10-12-2025\Objective 1\OB1.1\Dataset\HindiFlickr30K_release\verification_study"
)

OUTPUT_FILE = OUTPUT_DIR / "kappa_results.csv"

ID_COLUMN = "Caption ID"

CRITERIA = [
    "Visual_Relevance",
    "Semantic_Adequacy",
    "Grammaticality",
    "Fluency",
    "Factual_Consistency",
]

VALID_RATINGS = {1, 2, 3}
N_ANNOTATORS = 3

# ------------------------------------------------
# 2. NORMALIZE HEADINGS
# ------------------------------------------------

def normalize_heading(value):
    """
    Normalize case, spaces, punctuation, and invisible characters.
    For example:
        'Visual_Relevance' -> 'visualrelevance'
        ' Caption ID '     -> 'captionid'
    """
    if pd.isna(value):
        return ""

    value = unicodedata.normalize("NFKC", str(value))
    value = value.replace("\u200b", "").replace("\ufeff", "")
    return re.sub(r"[^a-z0-9]", "", value.casefold())


HEADER_ALIASES = {
    "captionid": ID_COLUMN,
    "visualrelevance": "Visual_Relevance",
    "semanticadequacy": "Semantic_Adequacy",
    "grammaticality": "Grammaticality",
    "fluency": "Fluency",
    "factualconsistency": "Factual_Consistency",
}

REQUIRED_FIELDS = [ID_COLUMN] + CRITERIA


def normalize_caption_id(value):
    """Make numeric IDs such as 1 and 1.0 comparable."""
    if pd.isna(value):
        return None

    if isinstance(value, (int, np.integer)):
        return str(int(value))

    if isinstance(value, (float, np.floating)):
        if np.isfinite(value) and float(value).is_integer():
            return str(int(value))

    result = str(value).strip()

    if not result:
        return None

    # Normalize text IDs such as '1.0' to '1'.
    if re.fullmatch(r"\d+\.0+", result):
        result = result.split(".")[0]

    return result


# ------------------------------------------------
# 3. FIND THE HEADER ROW ON ANY WORKSHEET
# ------------------------------------------------

def load_annotator_excel(filepath, annotator_name):

    excel = pd.ExcelFile(filepath, engine="openpyxl")
    candidates = []
    diagnostic = []

    for sheet_name in excel.sheet_names:

        raw = pd.read_excel(
            excel,
            sheet_name=sheet_name,
            header=None,
            dtype=object
        )

        # Keep a short diagnostic preview in case the header is absent.
        diagnostic.append(
            f"\nWorksheet: {sheet_name}\n"
            + raw.head(15).to_string(index=False, header=False)
        )

        # Search every row for all six required header labels.
        for row_index in range(len(raw)):

            found = {}

            for col_index, value in enumerate(raw.iloc[row_index].tolist()):
                normalized = normalize_heading(value)

                if normalized in HEADER_ALIASES:
                    canonical = HEADER_ALIASES[normalized]
                    found.setdefault(canonical, col_index)

            if all(field in found for field in REQUIRED_FIELDS):
                candidates.append({
                    "sheet_name": sheet_name,
                    "raw": raw,
                    "header_row": row_index,
                    "column_positions": found,
                })

    if not candidates:
        print(f"\nCould not locate the complete annotation header: {annotator_name}")
        print("Workbook:", filepath)
        print("Worksheets:", excel.sheet_names)
        print("\nFirst 15 rows of each worksheet:")

        for preview in diagnostic:
            print(preview)

        raise ValueError(
            f"\nNo worksheet in {filepath.name} contains a single row "
            "with Caption ID and all five rating headings.\n"
            "Check the worksheet previews above. The header may be split "
            "across rows, missing from the workbook, or stored as an "
            "image rather than editable Excel cell text."
        )

    # Use the first sheet containing the complete header.
    selected = candidates[0]

    sheet_name = selected["sheet_name"]
    raw = selected["raw"]
    header_row = selected["header_row"]
    positions = selected["column_positions"]

    # Extract the six columns by their actual positions.
    df = pd.DataFrame({
        field: raw.iloc[header_row + 1:, positions[field]].values
        for field in REQUIRED_FIELDS
    })

    # Remove entirely empty rows.
    df = df.dropna(how="all").copy()

    # Treat whitespace-only cells as missing.
    for field in REQUIRED_FIELDS:
        df[field] = df[field].map(
            lambda value: (
                np.nan
                if isinstance(value, str) and not value.strip()
                else value
            )
        )

    # Remove image-only/blank records, but do not silently discard
    # rating records whose Caption ID is missing.
    has_ratings = df[CRITERIA].notna().any(axis=1)
    has_id = df[ID_COLUMN].notna()

    orphan_rows = df[has_ratings & ~has_id]

    if not orphan_rows.empty:
        print(f"\nProblematic rows in {annotator_name}:")
        print(orphan_rows.head(10).to_string(index=False))

        raise ValueError(
            f"{annotator_name}: rating values were found on rows "
            "without a Caption ID. The data may be split over multiple "
            "rows. Arrange each caption ID and its five ratings on the "
            "same row before computing kappa."
        )

    # Ignore rows that contain only an embedded image or other content.
    df = df[has_id].copy()

    df[ID_COLUMN] = df[ID_COLUMN].map(normalize_caption_id)

    if df[ID_COLUMN].isna().any():
        raise ValueError(f"{annotator_name} has a blank Caption ID.")

    if df[ID_COLUMN].duplicated().any():
        duplicates = df.loc[
            df[ID_COLUMN].duplicated(keep=False), ID_COLUMN
        ].unique().tolist()

        raise ValueError(
            f"{annotator_name} has duplicate Caption IDs: "
            f"{duplicates[:10]}. Check whether the worksheet has "
            "multiple annotation rows per caption."
        )

    print(f"\n{annotator_name}:")
    print("  Worksheet:", sheet_name)
    print("  Header row in Excel:", header_row + 1)
    print("  Annotation records:", len(df))
    print("  Rating columns found:", ", ".join(CRITERIA))

    return df.reset_index(drop=True)


# ------------------------------------------------
# 4. CHECK INPUT FILES
# ------------------------------------------------

for annotator, filepath in PATHS.items():
    if not filepath.is_file():
        raise FileNotFoundError(
            f"Excel file not found for {annotator}:\n{filepath}"
        )

# ------------------------------------------------
# 5. LOAD ALL ANNOTATORS
# ------------------------------------------------

dfs = {
    annotator: load_annotator_excel(filepath, annotator)
    for annotator, filepath in PATHS.items()
}

# ------------------------------------------------
# 6. VERIFY CAPTION IDS
# ------------------------------------------------

id_sets = {
    annotator: set(df[ID_COLUMN])
    for annotator, df in dfs.items()
}

reference_ids = id_sets["Annotator1"]

if not all(ids == reference_ids for ids in id_sets.values()):

    print("\nCaption IDs do not match across annotators.")

    for annotator, ids in id_sets.items():
        print(f"\n{annotator}")
        print("  Missing IDs:", len(reference_ids - ids))
        print("  Additional IDs:", len(ids - reference_ids))

    raise ValueError(
        "Correct the Caption ID mismatch before calculating agreement."
    )

caption_ids = sorted(reference_ids)

indexed = {
    annotator: df.set_index(ID_COLUMN)
    for annotator, df in dfs.items()
}

print("\nCommon caption IDs:", len(caption_ids))

# ------------------------------------------------
# 7. CALCULATE FLEISS' KAPPA
# ------------------------------------------------

results = []

for criterion in CRITERIA:

    print("\n" + "=" * 65)
    print("Criterion:", criterion)
    print("=" * 65)

    ratings = pd.DataFrame({
        annotator: indexed[annotator]
        .reindex(caption_ids)[criterion]
        for annotator in PATHS
    }, index=caption_ids)

    numeric = ratings.apply(pd.to_numeric, errors="coerce")

    # Reject unexpected text rather than treating it as a missing score.
    nonnumeric = ratings.notna() & numeric.isna()

    if nonnumeric.any().any():
        examples = ratings.where(nonnumeric).stack().head(10)
        raise ValueError(
            f"Nonnumeric ratings for {criterion}:\n{examples}"
        )

    # Only scores 1, 2 and 3 are permitted.
    invalid = numeric.notna() & ~numeric.isin(VALID_RATINGS)

    if invalid.any().any():
        examples = numeric.where(invalid).stack().head(10)
        raise ValueError(
            f"Invalid scores for {criterion}; only 1, 2, 3 are allowed:\n"
            f"{examples}"
        )

    # Fleiss' kappa requires the same number of ratings per included item.
    complete = numeric.dropna(axis=0, how="any")

    total_records = len(numeric)
    used_records = len(complete)
    excluded_records = total_records - used_records

    print("Total caption records:", total_records)
    print("Complete records used:", used_records)
    print("Excluded incomplete records:", excluded_records)

    if used_records == 0:
        warnings.warn(f"No complete ratings for {criterion}.")

        results.append({
            "Criterion": criterion,
            "Total_Captions": total_records,
            "Complete_Captions": 0,
            "Excluded_Captions": excluded_records,
            "Observed_Agreement": np.nan,
            "Expected_Agreement": np.nan,
            "Fleiss_Kappa": np.nan,
            "Status": "Not calculated: no complete ratings",
        })
        continue

    values = complete.to_numpy(dtype=int)

    # Counts of ratings 1, 2, and 3 for each caption.
    # Every item must have exactly three ratings.
    matrix = np.column_stack([
        (values == score).sum(axis=1)
        for score in (1, 2, 3)
    ]).astype(int)

    if not np.all(matrix.sum(axis=1) == N_ANNOTATORS):
        raise ValueError(
            f"Incorrect number of ratings for {criterion}."
        )

    # Observed agreement.
    observed_by_item = (
        (matrix ** 2).sum(axis=1) - N_ANNOTATORS
    ) / (N_ANNOTATORS * (N_ANNOTATORS - 1))

    observed = float(observed_by_item.mean())

    # Expected agreement by chance.
    proportions = matrix.sum(axis=0) / (
        used_records * N_ANNOTATORS
    )

    expected = float(np.sum(proportions ** 2))

    if np.isclose(expected, 1.0):
        kappa = np.nan
        status = "Undefined: all ratings in one category"
    else:
        kappa = float(fleiss_kappa(matrix, method="fleiss"))

        status = (
            "Calculated"
            if np.isfinite(kappa)
            else "Undefined or non-finite"
        )

    print(f"Observed agreement: {observed:.4f}")
    print(f"Expected agreement: {expected:.4f}")
    print(
        f"Fleiss' kappa: {kappa:.4f}"
        if np.isfinite(kappa)
        else "Fleiss' kappa: undefined"
    )
    print("Status:", status)

    results.append({
        "Criterion": criterion,
        "Total_Captions": total_records,
        "Complete_Captions": used_records,
        "Excluded_Captions": excluded_records,
        "Observed_Agreement": observed,
        "Expected_Agreement": expected,
        "Fleiss_Kappa": kappa,
        "Status": status,
    })

# ------------------------------------------------
# 8. SAVE RESULTS
# ------------------------------------------------

results_df = pd.DataFrame(results)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

results_df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig",
    float_format="%.6f"
)

# Verify saved results.
check = pd.read_csv(OUTPUT_FILE, encoding="utf-8-sig")
assert len(check) == len(CRITERIA)

print("\n" + "=" * 65)
print("FINAL RESULTS")
print("=" * 65)
display(check)

print("\nResults successfully saved to:")
print(OUTPUT_FILE)

print(
    "\nNote: Fleiss' kappa treats scores 1, 2, and 3 as nominal "
    "categories. Because your scale is ordinal, consider reporting "
    "an appropriate ordinal agreement statistic as a complementary "
    "analysis in the journal manuscript."
)


Annotator1:
  Worksheet: Annotator1
  Header row in Excel: 1
  Annotation records: 500
  Rating columns found: Visual_Relevance, Semantic_Adequacy, Grammaticality, Fluency, Factual_Consistency

Annotator2:
  Worksheet: Annotator2
  Header row in Excel: 1
  Annotation records: 500
  Rating columns found: Visual_Relevance, Semantic_Adequacy, Grammaticality, Fluency, Factual_Consistency

Annotator3:
  Worksheet: Annotator3
  Header row in Excel: 1
  Annotation records: 500
  Rating columns found: Visual_Relevance, Semantic_Adequacy, Grammaticality, Fluency, Factual_Consistency

Common caption IDs: 500

Criterion: Visual_Relevance
Total caption records: 500
Complete records used: 500
Excluded incomplete records: 0
Observed agreement: 0.9113
Expected agreement: 0.4208
Fleiss' kappa: 0.8469
Status: Calculated

Criterion: Semantic_Adequacy
Total caption records: 500
Complete records used: 500
Excluded incomplete records: 0
Observed agreement: 0.9027
Expected agreement: 0.4179
Fleiss' kappa: 

,Criterion,Total_Captions,Complete_Captions,Excluded_Captions,Observed_Agreement,Expected_Agreement,Fleiss_Kappa,Status
0,Visual_Relevance,500,500,0,0.911333,0.420846,0.846903,Calculated
1,Semantic_Adequacy,500,500,0,0.902667,0.417860,0.832801,Calculated
2,Grammaticality,500,500,0,0.911333,0.413357,0.848857,Calculated
3,Fluency,500,500,0,0.906000,0.411046,0.840395,Calculated
4,Factual_Consistency,500,500,0,0.911333,0.411176,0.849417,Calculated



Results successfully saved to:
D:\DATASET On 10-12-2025\Objective 1\OB1.1\Dataset\HindiFlickr30K_release\verification_study\kappa_results.csv

Note: Fleiss' kappa treats scores 1, 2, and 3 as nominal categories. Because your scale is ordinal, consider reporting an appropriate ordinal agreement statistic as a complementary analysis in the journal manuscript.
